# Vision Zero – AGAHI Model Development Framework (Bhubaneswar Traffic Suite)
### End-to-End Traffic Conflict, Near-Miss Detection, and Model Fine-Tuning

Configured with your curated **100 Bhubaneswar Traffic Video Clips** for dataset extraction, multi-object tracking, and collision analytics.

## 1. Hardware Check & Mount Google Drive
Mounting Google Drive preserves checkpoints (`.pth`), extracted frames, and output videos permanently across Colab Free sessions.

In [ ]:
!nvidia-smi

from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/VisionZero_AGAHI"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "output")
VIDEOS_DIR = os.path.join(PROJECT_DIR, "raw_videos")
FRAMES_DIR = os.path.join(PROJECT_DIR, "extracted_frames")

for path in [PROJECT_DIR, CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR, VIDEOS_DIR, FRAMES_DIR]:
    os.makedirs(path, exist_ok=True)

print(f"\nAGAHI Workspace active at: {PROJECT_DIR}")

## 2. Install Dependencies
Installs `rfdetr`, `supervision`, `roboflow`, `yt-dlp`, `shapely`, and scientific visualization packages.

In [ ]:
!pip install -q "rfdetr[train]" supervision opencv-python-headless scipy matplotlib pandas roboflow shapely yt-dlp
print("All libraries installed successfully!")

## 3. Curated 100 Bhubaneswar Traffic Clips & Resilient Batch Downloader
This cell downloads 20-second snippets. It is built with **error-protection** (`--ignore-errors`), so if any single YouTube video is restricted or unavailable, it automatically skips it and moves to the next video without stopping!

In [ ]:
import os

BHUBANESWAR_URLS = [
    "https://www.youtube.com/watch?v=SahRJDv2fuk",
    "https://www.youtube.com/watch?v=ehWZq3YESeo",
    "https://www.youtube.com/watch?v=SbOsh6ch8LY",
    "https://www.youtube.com/watch?v=zgLtTu__WkE",
    "https://www.youtube.com/watch?v=eT6mcqi6fnQ",
    "https://www.youtube.com/watch?v=GXlGqpHZxtg",
    "https://www.youtube.com/watch?v=Mbjb5EGF7CA",
    "https://www.youtube.com/watch?v=POakSEWNrJw",
    "https://www.youtube.com/watch?v=Ll3KDlF5sRk",
    "https://www.youtube.com/watch?v=kwY2iWBYYT0",
    "https://www.youtube.com/watch?v=FmscD9ZKZzI",
    "https://www.youtube.com/watch?v=dKqlUo0uoEw",
    "https://www.youtube.com/watch?v=ZJNL3-epwgQ",
    "https://www.youtube.com/watch?v=LL4qAkQ5wz8",
    "https://www.youtube.com/watch?v=RWp-Y7MzeXo",
    "https://www.youtube.com/watch?v=twP6gcdb6-s",
    "https://www.youtube.com/watch?v=JYg8nz3vHlU",
    "https://www.youtube.com/watch?v=1xNxX8k7GHc",
    "https://www.youtube.com/watch?v=AoB7mfmQN_Q",
    "https://www.youtube.com/watch?v=s9yCnF_ZUDg",
    "https://www.youtube.com/watch?v=YNxyEWB-xvU",
    "https://www.youtube.com/watch?v=67hgF1K9wZM",
    "https://www.youtube.com/watch?v=TehswInbuPs",
    "https://www.youtube.com/watch?v=llh4BYaZuMI",
    "https://www.youtube.com/watch?v=IvmoeXBOUrA",
    "https://www.youtube.com/watch?v=9HOZ6jevuko",
    "https://www.youtube.com/watch?v=YYSq78xPYfA",
    "https://www.youtube.com/watch?v=HL7dSd88vKc",
    "https://www.youtube.com/watch?v=1_3M5cN5Vuk",
    "https://www.youtube.com/watch?v=V_7DSrbYRTU",
    "https://www.youtube.com/watch?v=RzRRKrFo15M",
    "https://www.youtube.com/watch?v=csedb5H9bTE",
    "https://www.youtube.com/watch?v=LIzPnDoRO00",
    "https://www.youtube.com/watch?v=RziUQJKA94E",
    "https://www.youtube.com/watch?v=uwpZC6JwItA",
    "https://www.youtube.com/watch?v=zh_n88laDk8",
    "https://www.youtube.com/watch?v=Oi1xWYdraq0",
    "https://www.youtube.com/watch?v=f3CKJ-Sk--g",
    "https://www.youtube.com/watch?v=LJ9_kF2FpsY",
    "https://www.youtube.com/watch?v=xEe5IuR-WZM",
    "https://www.youtube.com/watch?v=z903-74qfVs",
    "https://www.youtube.com/watch?v=khMGy6rrZe4",
    "https://www.youtube.com/watch?v=xQ1SUcIUNbM",
    "https://www.youtube.com/watch?v=M_JA-3mwCtI",
    "https://www.youtube.com/watch?v=uAtXeLZDv10",
    "https://www.youtube.com/watch?v=p6H1M6S5lbk",
    "https://www.youtube.com/watch?v=BaP-M4MRB6U",
    "https://www.youtube.com/watch?v=3p0jcTDqnC8",
    "https://www.youtube.com/watch?v=b0QlkwHOT48",
    "https://www.youtube.com/watch?v=tsrNRY8KN_Y",
    "https://www.youtube.com/watch?v=PvNmv-AQlZI",
    "https://www.youtube.com/watch?v=S_MnuCKQH1s",
    "https://www.youtube.com/watch?v=oKJ6egBNZ7Y",
    "https://www.youtube.com/watch?v=KESozjF_pEo",
    "https://www.youtube.com/watch?v=pnzrUfoEuVc",
    "https://www.youtube.com/watch?v=P1cybFuO6_E",
    "https://www.youtube.com/watch?v=ham_ccByGKw",
    "https://www.youtube.com/watch?v=vdDpTFVQOnQ",
    "https://www.youtube.com/watch?v=FuC5p1Dwngk",
    "https://www.youtube.com/watch?v=SEywFLm448A",
    "https://www.youtube.com/watch?v=0x_a8yueixY",
    "https://www.youtube.com/watch?v=xu9Otp82BVU",
    "https://www.youtube.com/watch?v=dEBM-btwya8",
    "https://www.youtube.com/watch?v=hOvIduODe4c",
    "https://www.youtube.com/watch?v=8N4Jz2DU2pA",
    "https://www.youtube.com/watch?v=NtYB9_Tk5LA",
    "https://www.youtube.com/watch?v=hx-d-FkGCp0",
    "https://www.youtube.com/watch?v=eL0WyDKCVL0",
    "https://www.youtube.com/watch?v=kK7mbBqpTmM",
    "https://www.youtube.com/watch?v=8Ygs72gSkYs",
    "https://www.youtube.com/watch?v=A7cuy5PTNyE",
    "https://www.youtube.com/watch?v=KjHKn_8jeO8",
    "https://www.youtube.com/watch?v=89ElPgJlmoM",
    "https://www.youtube.com/watch?v=JFye92Gi3cw",
    "https://www.youtube.com/watch?v=584IzhJjl44",
    "https://www.youtube.com/watch?v=fXLSc-8HHQ4",
    "https://www.youtube.com/watch?v=lk0Qw6myi2U",
    "https://www.youtube.com/watch?v=1709DZPfzRA",
    "https://www.youtube.com/watch?v=bufPlW_bDUw",
    "https://www.youtube.com/watch?v=HAR10VMLekg",
    "https://www.youtube.com/watch?v=5lSWmIraU0E",
    "https://www.youtube.com/watch?v=FEPoN9FXu5A",
    "https://www.youtube.com/watch?v=cBvXjAev4Io",
    "https://www.youtube.com/watch?v=LsIsTejf8yQ",
    "https://www.youtube.com/watch?v=Z3iijHbLJlw",
    "https://www.youtube.com/watch?v=ZzCB0Ci1AO0",
    "https://www.youtube.com/watch?v=qHwwCiWxCwU",
    "https://www.youtube.com/watch?v=VqQQ8IYbB8o",
    "https://www.youtube.com/watch?v=upq0agxBTqc",
    "https://www.youtube.com/watch?v=1QsK-v-xJ-Y",
    "https://www.youtube.com/watch?v=JX6GZPmAUtY",
    "https://www.youtube.com/watch?v=bwTEzKiv-dw",
    "https://www.youtube.com/watch?v=S2D-oghT0Ds",
    "https://www.youtube.com/watch?v=uLxze0goHlA",
    "https://www.youtube.com/watch?v=eX-oenEKhM4",
    "https://www.youtube.com/watch?v=B83czTRb_pk",
    "https://www.youtube.com/watch?v=cPZEvy-cxsY",
    "https://www.youtube.com/watch?v=3ScV2QkMMpI",
    "https://www.youtube.com/watch?v=VxB6RvE6Fnw",
    "https://www.youtube.com/watch?v=XbDqlOkUQFU"
]

# Set how many clips you want to download (e.g. 10 to start, or len(BHUBANESWAR_URLS) for all 100):
NUM_CLIPS_TO_DOWNLOAD = 10

LOCAL_VIDEOS_FOLDER = "/content/bhubaneswar_clips"
os.makedirs(LOCAL_VIDEOS_FOLDER, exist_ok=True)

downloaded_files = []
print(f"Downloading {NUM_CLIPS_TO_DOWNLOAD} clips...")

for idx in range(min(NUM_CLIPS_TO_DOWNLOAD, len(BHUBANESWAR_URLS))):
    url = BHUBANESWAR_URLS[idx]
    out_path = os.path.join(LOCAL_VIDEOS_FOLDER, f"bhubaneswar_clip_{idx+1:03d}.mp4")
    
    if not os.path.exists(out_path) or os.path.getsize(out_path) < 1000:
        print(f"[{idx+1}/{NUM_CLIPS_TO_DOWNLOAD}] Downloading: {url}")
        try:
            # Try downloading 20s section with resilient format selection
            !yt-dlp --ignore-errors --no-warnings -f "best[ext=mp4]/best" --download-sections "*00:10-00:30" -o "{out_path}" "{url}"
        except Exception as e:
            print(f"[Warning] Skipped clip {idx+1} due to error: {e}")
    else:
        print(f"[{idx+1}/{NUM_CLIPS_TO_DOWNLOAD}] Already exists: {out_path}")
    
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        downloaded_files.append(out_path)

# Fallback if needed
if not downloaded_files:
    fallback_url = "https://github.com/roboflow/supervision/raw/main/examples/traffic_analysis/traffic_analysis.mp4"
    fallback_path = "/content/fallback_traffic.mp4"
    !wget -q -O {fallback_path} "{fallback_url}"
    downloaded_files.append(fallback_path)

ACTIVE_VIDEO = downloaded_files[0]
print(f"\nDownload finished! {len(downloaded_files)} playable video(s) ready.")
print(f"Active video for detection & tracking: {ACTIVE_VIDEO}")

## 4. Multi-Clip Frame Harvester (Builds Dataset from All Clips)
Extracts 5 diverse frames per video across your downloaded clips and zips them directly into Google Drive for Roboflow annotation.

In [ ]:
import cv2
import shutil

frames_output_dir = "/content/dataset_frames_harvested"
os.makedirs(frames_output_dir, exist_ok=True)

total_harvested = 0
for vid_path in downloaded_files:
    vid_name = os.path.splitext(os.path.basename(vid_path))[0]
    cap = cv2.VideoCapture(vid_path)
    total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_f > 10:
        step = max(1, total_f // 5)
        f_idx = 0
        saved_in_vid = 0
        while cap.isOpened() and saved_in_vid < 5:
            ret, frame = cap.read()
            if not ret:
                break
            if f_idx % step == 0:
                out_name = os.path.join(frames_output_dir, f"{vid_name}_f{saved_in_vid:02d}.jpg")
                cv2.imwrite(out_name, frame)
                saved_in_vid += 1
                total_harvested += 1
            f_idx += 1
    cap.release()

print(f"Harvested {total_harvested} high-diversity frames into {frames_output_dir}")
zip_path = os.path.join(PROJECT_DIR, "bhubaneswar_annotator_frames")
shutil.make_archive(zip_path, 'zip', frames_output_dir)
print(f"Zipped dataset frames saved permanently in Google Drive: {zip_path}.zip")

## 5. Visual Point-Picker Road Calibration Tool
Displays the first frame of your active clip with a cyan grid every 100 pixels.
Read the `(X, Y)` for 4 corners of the road surface (`Top-Left`, `Top-Right`, `Bottom-Right`, `Bottom-Left`).

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(ACTIVE_VIDEO)
ret, frame = cap.read()
cap.release()

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w, _ = frame_rgb.shape
    
    plt.figure(figsize=(14, 8))
    plt.imshow(frame_rgb)
    plt.grid(True, color='cyan', linestyle='--', linewidth=0.6, alpha=0.7)
    plt.xticks(range(0, w, 100))
    plt.yticks(range(0, h, 100))
    plt.title(f"Calibration Grid ({w}x{h}) - Read 4 Road Corner Points (X, Y)", fontsize=14, weight='bold')
    plt.show()
else:
    print("Could not read video frame.")

## 6. Safety Analytics Engine (TTC + PET + Homography)
- **Homography**: Transforms 2D camera pixels $(u, v)$ to real-world ground meters $(X, Y)$
- **Speed**: $\|V\| = \sqrt{\Delta X^2 + \Delta Y^2} / \Delta t \times 3.6 \text{ km/h}$
- **TTC (Time-To-Collision)**: Approaching collision estimation: $\text{TTC} = \frac{\Delta \text{distance}}{v_{\text{closing}}}$
- **PET (Post-Encroachment Time)**: Crossing conflict time gap: $\text{PET} = t_{\text{enter}}^B - t_{\text{exit}}^A$

In [ ]:
import cv2
import numpy as np
from collections import defaultdict, deque
from shapely.geometry import Polygon, Point

class ViewTransformer:
    def __init__(self, source_quadrilateral: np.ndarray, target_rectangle: np.ndarray):
        self.H = cv2.getPerspectiveTransform(
            source_quadrilateral.astype(np.float32),
            target_rectangle.astype(np.float32)
        )

    def transform_points(self, points: np.ndarray) -> np.ndarray:
        if len(points) == 0:
            return np.empty((0, 2))
        reshaped = points.reshape(-1, 1, 2).astype(np.float32)
        transformed = cv2.perspectiveTransform(reshaped, self.H)
        return transformed.reshape(-1, 2)


class SafetyAnalyticsEngine:
    def __init__(self, transformer: ViewTransformer, fps: float = 30.0, max_history: int = 20, conflict_polygon=None):
        self.transformer = transformer
        self.fps = fps
        self.dt = 1.0 / fps
        self.max_history = max_history
        self.history = defaultdict(lambda: deque(maxlen=self.max_history))
        self.conflict_events = []
        self.conflict_zone = Polygon(conflict_polygon) if conflict_polygon is not None else None
        self.zone_presence = {}
        self.last_zone_exit = None

    def update_tracks(self, tracker_ids, xyxy_boxes, frame_idx):
        timestamp = frame_idx * self.dt
        bottom_centers = np.column_stack([
            (xyxy_boxes[:, 0] + xyxy_boxes[:, 2]) / 2,
            xyxy_boxes[:, 3]
        ])
        ground_meters = self.transformer.transform_points(bottom_centers)
        
        current_data = {}
        for trk_id, center_px, pos_m in zip(tracker_ids, bottom_centers, ground_meters):
            self.history[trk_id].append((timestamp, pos_m, center_px))
            speed_kmh, vel_vector = self._compute_velocity(trk_id)
            
            in_zone = False
            if self.conflict_zone is not None:
                in_zone = self.conflict_zone.contains(Point(center_px[0], center_px[1]))
                self._update_pet_state(trk_id, in_zone, timestamp, center_px, pos_m)

            current_data[trk_id] = {
                'pos_m': pos_m,
                'center_px': center_px,
                'speed_kmh': speed_kmh,
                'vel_vector': vel_vector,
                'in_zone': in_zone
            }
        return current_data

    def _compute_velocity(self, trk_id):
        hist = self.history[trk_id]
        if len(hist) < 3:
            return 0.0, np.array([0.0, 0.0])
        t0, p0, _ = hist[0]
        tn, pn, _ = hist[-1]
        time_elapsed = tn - t0
        if time_elapsed <= 0:
            return 0.0, np.array([0.0, 0.0])
        vel_m_s = (pn - p0) / time_elapsed
        speed_kmh = np.linalg.norm(vel_m_s) * 3.6
        return speed_kmh, vel_m_s

    def _update_pet_state(self, trk_id, in_zone, timestamp, center_px, pos_m):
        if trk_id not in self.zone_presence:
            if in_zone:
                self.zone_presence[trk_id] = {'entered': timestamp, 'exited': None, 'active': True}
                if self.last_zone_exit is not None:
                    prev_id, prev_exit_t, prev_pt_px = self.last_zone_exit
                    if prev_id != trk_id:
                        pet = timestamp - prev_exit_t
                        if 0.1 <= pet < 1.8:
                            self.conflict_events.append({
                                'type': 'PET',
                                'frame': int(timestamp * self.fps),
                                'id1': prev_id,
                                'id2': trk_id,
                                'metric_val': round(float(pet), 2),
                                'px': center_px.tolist(),
                                'meter': pos_m.tolist()
                            })
        else:
            record = self.zone_presence[trk_id]
            if record['active'] and not in_zone:
                record['exited'] = timestamp
                record['active'] = False
                self.last_zone_exit = (trk_id, timestamp, center_px)

    def compute_ttc_conflicts(self, current_data, frame_idx, ttc_threshold=1.6, max_distance_m=12.0):
        conflicts = []
        ids = list(current_data.keys())
        n = len(ids)
        for i in range(n):
            for j in range(i + 1, n):
                id1, id2 = ids[i], ids[j]
                d1, d2 = current_data[id1], current_data[id2]
                p1, p2 = d1['pos_m'], d2['pos_m']
                v1, v2 = d1['vel_vector'], d2['vel_vector']
                dist = np.linalg.norm(p2 - p1)
                if dist < 0.2 or dist > max_distance_m:
                    continue
                v_rel = v1 - v2
                unit_r = (p2 - p1) / dist
                v_closing = np.dot(v_rel, unit_r)
                if v_closing > 0.8:
                    ttc = dist / v_closing
                    if ttc < ttc_threshold:
                        conflicts.append((id1, id2, ttc, d1['center_px'], d2['center_px']))
                        self.conflict_events.append({
                            'type': 'TTC',
                            'frame': frame_idx,
                            'id1': id1,
                            'id2': id2,
                            'metric_val': round(float(ttc), 2),
                            'dist_m': round(float(dist), 2),
                            'px': ((d1['center_px'] + d2['center_px']) / 2).tolist(),
                            'meter': ((p1 + p2) / 2).tolist()
                        })
        return conflicts

print("Safety Analytics Engine ready!")

## 7. Run Detection + ByteTrack + Near-Miss HUD Pipeline
Processes the active video with RF-DETR and ByteTrack, calculating speeds and flagging TTC/PET near-miss conflicts.

In [ ]:
import cv2
import supervision as sv
from rfdetr import RFDETRMedium
from rfdetr.assets.coco_classes import COCO_CLASSES
import numpy as np

print("Loading RF-DETR Medium model...")
model = RFDETRMedium()

tracker = sv.ByteTrack(track_activation_threshold=0.35, lost_track_buffer=45)
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.45, text_padding=3)
trace_annotator = sv.TraceAnnotator(trace_length=35, thickness=2)

# Read active video dimensions
video_info = sv.VideoInfo.from_video_path(ACTIVE_VIDEO)
w, h = video_info.width, video_info.height

# Adaptive road perspective quadrilateral (adjust in Cell 5 if needed)
SOURCE_POLYGON = np.array([
    [int(w * 0.35), int(h * 0.35)],
    [int(w * 0.65), int(h * 0.35)],
    [int(w * 0.95), int(h * 0.95)],
    [int(w * 0.05), int(h * 0.95)]
], dtype=np.float32)

TARGET_RECTANGLE = np.array([
    [0, 0], [10, 0], [10, 30], [0, 30]
], dtype=np.float32)

transformer = ViewTransformer(SOURCE_POLYGON, TARGET_RECTANGLE)
output_video = os.path.join(OUTPUT_DIR, "bhubaneswar_tracked_output.mp4")
analytics = SafetyAnalyticsEngine(transformer=transformer, fps=video_info.fps, conflict_polygon=SOURCE_POLYGON)

print(f"Processing {ACTIVE_VIDEO} ({video_info.total_frames} frames)...")

with sv.VideoSink(target_path=output_video, video_info=video_info) as sink:
    frame_idx = 0
    for frame in sv.get_video_frames_generator(source_path=ACTIVE_VIDEO):
        frame_idx += 1
        
        # Predict using RGB frame
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        detections = model.predict(frame_rgb, threshold=0.35)
        detections = tracker.update_with_detections(detections)
        
        annotated_frame = frame.copy()
        cv2.polylines(annotated_frame, [SOURCE_POLYGON.astype(np.int32)], isClosed=True, color=(255, 200, 0), thickness=1)

        if len(detections) > 0 and detections.tracker_id is not None:
            current_data = analytics.update_tracks(detections.tracker_id, detections.xyxy, frame_idx)
            conflicts = analytics.compute_ttc_conflicts(current_data, frame_idx, ttc_threshold=1.6)
            
            labels = []
            for trk_id, cls_id in zip(detections.tracker_id, detections.class_id):
                cls_name = COCO_CLASSES[cls_id] if cls_id < len(COCO_CLASSES) else "vehicle"
                speed = current_data[trk_id]['speed_kmh'] if trk_id in current_data else 0.0
                labels.append(f"#{trk_id} {cls_name} | {speed:.0f} km/h")

            annotated_frame = trace_annotator.annotate(scene=annotated_frame, detections=detections)
            annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
            annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
            
            for id1, id2, ttc, p1_px, p2_px in conflicts:
                pt1 = (int(p1_px[0]), int(p1_px[1]))
                pt2 = (int(p2_px[0]), int(p2_px[1]))
                cv2.line(annotated_frame, pt1, pt2, (0, 0, 255), 3)
                mid_x = int((pt1[0] + pt2[0]) / 2)
                mid_y = int((pt1[1] + pt2[1]) / 2) - 10
                cv2.putText(annotated_frame, f"TTC ALERT: {ttc:.1f}s", (mid_x - 55, mid_y),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2, cv2.LINE_AA)

        cv2.rectangle(annotated_frame, (10, 10), (380, 45), (30, 30, 30), -1)
        cv2.putText(annotated_frame, f"AGAHI Vision Zero | Conflicts: {len(analytics.conflict_events)}", 
                    (20, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)
        sink.write_frame(frame=annotated_frame)
        if frame_idx % 60 == 0:
            print(f"Processed {frame_idx}/{video_info.total_frames} frames...")

print(f"\nProcessing complete! Video saved in Google Drive: {output_video}")
print(f"Total near-miss events detected: {len(analytics.conflict_events)}")

## 8. Embedded Video Preview
Watch the output directly in Google Colab.

In [ ]:
from IPython.display import HTML
from base64 import b64encode

preview_file = "/content/display_preview.mp4"
!ffmpeg -y -i "{output_video}" -vcodec libx264 -crf 26 "{preview_file}" -loglevel quiet

mp4_bytes = open(preview_file, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
HTML(f"""
<video width=750 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")

## 9. Risk Heatmap Generation (Kernel Density Estimation)
Generates the intersection risk heatmap displaying dangerous conflict clusters.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
import pandas as pd

conflicts = analytics.conflict_events
if len(conflicts) >= 2:
    df_conflicts = pd.DataFrame(conflicts)
    pts = np.array(df_conflicts['px'].tolist())
    x = pts[:, 0]
    y = pts[:, 1]
    xy = np.vstack([x, y])
    kde = gaussian_kde(xy)
    
    cap = cv2.VideoCapture(ACTIVE_VIDEO)
    ret, bg_frame = cap.read()
    cap.release()
    bg_frame = cv2.cvtColor(bg_frame, cv2.COLOR_BGR2RGB)
    h, w, _ = bg_frame.shape
    
    xi, yi = np.mgrid[0:w:100j, 0:h:100j]
    zi = kde(np.vstack([xi.flatten(), yi.flatten()])).reshape(xi.shape)
    
    plt.figure(figsize=(12, 7))
    plt.imshow(bg_frame)
    plt.contourf(xi, yi, zi, levels=14, cmap='jet', alpha=0.55)
    plt.scatter(x, y, color='red', s=45, edgecolor='white', label='Conflict Event (TTC/PET)')
    plt.title("AGAHI Vision Zero – Intersection Conflict Risk Heatmap (KDE)", fontsize=14, weight='bold')
    plt.axis('off')
    plt.legend(loc='upper right')
    
    heatmap_path = os.path.join(OUTPUT_DIR, "risk_heatmap.png")
    plt.savefig(heatmap_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Risk heatmap saved to Google Drive: {heatmap_path}")
else:
    print(f"Found {len(conflicts)} conflict(s). Test with a busier video or longer duration.")

## 10. Starter COCO Dataset Generator for Indian Traffic Classes
Initializes valid COCO annotations for Indian vehicle types (`auto_rickshaw`, `motorcycle`, `car`, `bus`, `truck`, `pedestrian`).

In [ ]:
import json
import os
import cv2

train_img_dir = os.path.join(DATASET_DIR, "train", "images")
valid_img_dir = os.path.join(DATASET_DIR, "valid", "images")
os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(valid_img_dir, exist_ok=True)

INDIAN_CLASSES = [
    {"id": 1, "name": "car"},
    {"id": 2, "name": "auto_rickshaw"},
    {"id": 3, "name": "motorcycle"},
    {"id": 4, "name": "bus"},
    {"id": 5, "name": "truck"},
    {"id": 6, "name": "pedestrian"}
]

train_json_path = os.path.join(DATASET_DIR, "train", "_annotations.coco.json")
if not os.path.exists(train_json_path):
    print("Seeding COCO annotations from active video frames...")
    cap = cv2.VideoCapture(ACTIVE_VIDEO)
    coco_train = {"images": [], "annotations": [], "categories": INDIAN_CLASSES}
    ann_id = 1
    for i in range(15):
        ret, frame = cap.read()
        if not ret:
            break
        h, w, _ = frame.shape
        img_name = f"train_frame_{i:03d}.jpg"
        cv2.imwrite(os.path.join(train_img_dir, img_name), frame)
        coco_train["images"].append({"id": i+1, "file_name": img_name, "width": w, "height": h})
        coco_train["annotations"].append({
            "id": ann_id, "image_id": i+1, "category_id": 2,
            "bbox": [int(w*0.3), int(h*0.4), 120, 140], "area": 120*140, "iscrowd": 0
        })
        ann_id += 1
    cap.release()
    with open(train_json_path, 'w') as f:
        json.dump(coco_train, f, indent=2)
    
    valid_json_path = os.path.join(DATASET_DIR, "valid", "_annotations.coco.json")
    with open(valid_json_path, 'w') as f:
        json.dump(coco_train, f, indent=2)
    print(f"Starter COCO dataset ready at: {DATASET_DIR}")
else:
    print(f"Existing COCO dataset detected at: {DATASET_DIR}")

## 11. Fine-Tune RF-DETR for Bhubaneswar Traffic
Fine-tunes the transformer detector on Indian road classes. Saves checkpoints directly into Google Drive.

In [ ]:
from rfdetr import RFDETRMedium

train_json = os.path.join(DATASET_DIR, "train", "_annotations.coco.json")
if os.path.exists(train_json):
    print("Starting RF-DETR fine-tuning on Indian traffic dataset...")
    model = RFDETRMedium()
    model.train(
        dataset_dir=DATASET_DIR,
        epochs=15,
        batch_size=4,
        lr=0.0001,
        output_dir=CHECKPOINT_DIR
    )
    print(f"\nTraining complete! Checkpoints safely saved in Google Drive: {CHECKPOINT_DIR}")
else:
    print(f"Dataset not found at {DATASET_DIR}.")

## 12. Inference with Fine-Tuned Checkpoint
Load your trained weights to detect auto-rickshaws, dense two-wheelers, and pedestrians on Indian intersections.

In [ ]:
best_checkpoint = os.path.join(CHECKPOINT_DIR, "checkpoint_best_total.pth")

if os.path.exists(best_checkpoint):
    print(f"Loading custom fine-tuned weights from: {best_checkpoint}")
    custom_model = RFDETRMedium(pretrain_weights=best_checkpoint)
    print("Custom AGAHI model ready for production inference!")
else:
    print(f"No checkpoint found at {best_checkpoint} yet. Run Cell 11 training first.")